# Run 6: Jev and SAM 3 with the trowel's geometry, held where it balances

Runs 4 and 5 take the trowel by its handle from the camera alone. Lifted there, it tilts: the steel
blade outweighs the beech handle, and the handle grip sits 56 mm from the tool's centre of mass. With
the trowel's STL (`piper_llm/assets/objects/trowel`), the pose comes from matching the geometry's
outline to SAM 3's mask, and the grasp is searched on the posed geometry: the line between the pads
passes as close to the centre of mass as the fingers can grip, straight across the ferrule, so the
trowel is carried level. The centre of mass is not the middle of the volume: `meta.json` weighs the
steel parts against the wooden handle.


# ⚠️ Before you run this

- E-stop within reach.
- Nobody inside the arm's reach.
- Table clear of anything breakable.
- `speed_percent` at 10 for the first run.

The model can pick a wrong skill and perception can be wrong. The checks in
`piper_llm/safety.py` reduce the risk; they do not remove it.


In [ ]:
import os, numpy as np
from dotenv import load_dotenv           # pip install python-dotenv
load_dotenv()                            # reads .env, which is not in git

from piper_llm.arm import Arm
from piper_llm.camera import RealSense, Frame
from piper_llm.kinematics import Kinematics
from piper_llm.safety import Governor
from piper_llm.skills import Runner, PICK_PLACE_SKILLS
from piper_llm.task import PickPlace
from piper_llm.config import ArmConfig, SceneConfig, TROWEL_SCENE
from piper_llm.deciders import Jev
from piper_llm.perception import Detector
from piper_llm.record import Recorder
from piper_llm.geometry import ObjectGeometry


In [ ]:
# Dry check: no torque. Read the arm, camera and models first.
cam = RealSense()
rgb, depth = cam.frames()
print("camera ok", rgb.shape, "depth range %.2f-%.2f m" % (depth[depth>0].min(), depth.max()))

frame = Frame(cam)                        # needs calibration.json
kin = Kinematics()                        # needs the Menagerie MJCF
print("kinematics ok")


In [ ]:
# Read-only arm check. The arm does not move in this cell.
arm = Arm(ArmConfig(speed_percent=10, gripper_effort_nm=2.0))   # a smooth steel ferrule: the gripper's full squeeze
print("joints", np.round(arm.joints(), 3))
print("gripper", round(arm.gripper(), 2))
print("fault:", arm.status_error())


## Check perception before moving

The detections should sit on the trowel and the tray, the geometry's outline should match the trowel,
and the grasp should be across the ferrule. SAM 3 takes one phrase per pass, about 200 ms each on
this GPU after warm-up.


In [ ]:
detector = Detector(backend="sam3")
rgb, depth = cam.frames()
scene = SceneConfig(**TROWEL_SCENE)
for phrase in (scene.target, scene.place_target):
    d = detector.best(rgb, phrase, max_px=scene.target_max_px if phrase == scene.target else scene.place_max_px)
    if d is None:
        print(phrase, "NOT FOUND")
        continue
    p = frame.to_base(d.centre[0], d.centre[1], depth)
    print("%-10s score %.2f  base xyz %s" % (phrase, d.score, np.round(p, 3)))
print("detector median %.0f ms" % detector.median_ms())
geometry = ObjectGeometry.load("../piper_llm/assets/objects/trowel")
d = detector.best(rgb, scene.target, max_px=scene.target_max_px)
pose, fit, rest = geometry.locate(d.mask, frame)
point, yaw, width = geometry.grasp(pose, lowest_tool=scene.grasp_z)
print("trowel located: outline match %.2f, blade down: %s, centre of mass %s"
      % (fit, pose[2, 2] > 0.9, np.round(pose[:3, :3] @ geometry.com + pose[:3, 3], 3)))
print("grasp at %s, %.0f mm across; the centre of mass is %.1f mm from the line between the pads"
      % (np.round(point, 3), 1000 * width, 1000 * geometry.lever(pose, point, yaw)))


## Run

Each step: the task helper reads the camera and builds the state, Jev picks a skill, the governor
checks it, the runner moves. The grasp is planned while the target is in clear view and kept while the
fingers hide it; the tray is remembered while the arm is over it. Heights follow the object: it is
carried as far above the tray's rim as it can reach below the grasp, and lowered until that depth ends
just above the tray's floor. See "How a grasp works" in the README.

With the geometry, the grasp and the reach come from the trowel's STL posed on the table, not from depth,
and the grasp is where the trowel balances.


In [ ]:
rec = Recorder(cam, arm, "run6")       # colour, depth, joints and decisions to out/recordings
arm.home()
gov = Governor()
runner = Runner(arm=arm, kin=kin, gov=gov, scene=scene)
task = PickPlace(scene, detector, frame, runner, geometry=geometry)   # pose, grasp and reach from the STL
jev = Jev()

INSTRUCTIONS = ("Pick the next skill for a PiPER arm that must put the trowel in the tray. Each skill "
                "says when it applies: pick the one whose condition matches the state. Order: "
                "approach, descend, close, lift, move over, lower, open, retreat, done.")

gov.reset()
for step in range(25):
    rgb, depth = cam.frames()
    pose = kin.tool_pose(arm.joints())
    state = task.observe(rgb, depth, pose, arm)
    if task.problem:                      # found before anything is lifted
        print("cannot place it:", task.problem)
        rec.note("cannot place it: " + task.problem)
        break
    d = jev.choose(state, PICK_PLACE_SKILLS, INSTRUCTIONS)
    print(step, d.skill, "confidence %.2f" % (d.confidence or 0))
    rec.note("%d %s %.2f" % (step, d.skill, d.confidence or 0))

    ok, why = gov.check_confidence(d.skill, d.confidence)
    if ok:
        ok, why = gov.check_precondition(d.skill, state)
    if not ok:
        print("   refused:", why)
        rec.note("refused: " + why)
        continue
    if d.skill == "done":
        break
    target, yaw, grip = task.target(d.skill, pose)
    if target is None:
        print("   no target")
        continue
    reached, why = runner.move_tool_to(d.skill, target, yaw, grip)
    if why:
        print("   stopped:", why)
        rec.note("stopped: " + why)
    task.after(d.skill, reached, state, pose)

print("safety events:", gov.summary())
print("the object reaches %.0f cm below the grasp: carried at %.0f cm, released at %.0f cm"
      % (100 * task.hang, 100 * task.carry_z, 100 * task.release_z))
print("jev calls:", jev.calls, "median %.0f ms" % (1000*jev.seconds/max(jev.calls,1)),
      "cost $%.4f" % jev.cost)


In [ ]:
task.finish(runner, arm)   # leaves the gripper empty: places the object or puts it back
arm.close()
rec.close()
cam.close()
